# Tibeb Amharic lab: QLoRA on a free Kaggle GPU

Trains a small open model to be better at Amharic, then measures the gain. The same recipe is meant to scale to Tibeb-Pro once compute is available.

**Before running (right-hand panel, Settings):**
1. **Accelerator:** GPU T4 x2 or GPU P100 (one GPU is used).
2. **Internet:** On.
3. **Optional, your own sources:** upload your dictionary/grammar files as a Kaggle dataset named `amharic-sources` and attach it (Add Input). Accepted: `dictionary*.csv`, `conjugations*.csv`, `qa*.csv` (see `templates/`), and grammar `.txt`, `.docx` or `.pdf` (Unicode Amharic only).
4. **Optional, saving to Hugging Face:** Add-ons, Secrets, add `HF_TOKEN` (a write token) to upload the trained adapter.

**Time budget:** the defaults below should fit in one 12-hour session on a T4. Lower `PRETRAIN_STEPS` / `SFT_STEPS` for a quicker first run.

In [ ]:
%%capture
!pip install -U transformers peft trl bitsandbytes accelerate datasets huggingface_hub pypdf python-docx

In [ ]:
# ---- Settings ----
REPO = "https://github.com/reKidane/DeepSeek-V3"
BRANCH = "main"
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"   # text-only, Apache-2.0, fits a T4 in 4-bit
TOKENIZERS_TO_COMPARE = [
    BASE_MODEL,
    "Qwen/Qwen3.5-4B",
    "google/gemma-4-E4B-it",
    "rasyosef/Llama-3.2-1B-Amharic-Instruct",
    "deepseek-ai/DeepSeek-V4-Pro-0813",          # Tibeb-Pro's tokenizer
]
MAX_WEB_DOCS = 50_000     # FineWeb-2 Amharic documents to sample
EVAL_LIMIT = 200          # items per benchmark (None = full test sets, slower)
PRETRAIN_STEPS = 300      # stage A: continued training on Amharic text
SFT_STEPS = 300           # stage B: instruction tuning on Q&A pairs
HUB_ADAPTER_REPO = "rekidane/Tibeb-Amharic-lab-lora"   # private upload target

import os
os.chdir("/kaggle/working")
!rm -rf tibeb && git clone -q --depth 1 -b {BRANCH} {REPO} tibeb
os.chdir("/kaggle/working/tibeb/training/amharic")
!ls

## 1. Tokenizer efficiency on Amharic
Tokens per Amharic word (lower is better). High values mean Amharic is slow and expensive for that model; this also shows where Tibeb-Pro's tokenizer stands.

In [ ]:
import json
import pandas as pd
tok_args = " ".join(TOKENIZERS_TO_COMPARE)
!python eval_amharic.py --fertility-only --tokenizers {tok_args} --out results/fertility.json
pd.DataFrame(json.load(open("results/fertility.json"))).T

## 2. Build the training data
Your own sources (if attached) are converted first, then merged with Amharic Wikipedia, FineWeb-2 Amharic and the Amharic part of the Aya dataset. Anything overlapping the evaluation sets is removed.

In [ ]:
import glob, subprocess
src = "/kaggle/input/amharic-sources"
extra = []
if os.path.isdir(src):
    files = glob.glob(f"{src}/**/*", recursive=True)
    docs = [f for f in files if f.lower().endswith((".txt", ".docx", ".pdf"))]
    if docs:
        subprocess.run(["python", "extract_text.py", *docs, "--out-dir", "extracted"])
    def pick(key):
        return [f for f in files if os.path.basename(f).lower().startswith(key)
                and f.lower().endswith((".csv", ".tsv"))]
    cmd = ["python", "build_pairs.py", "--out", "data"]
    if pick("dictionary"):
        cmd += ["--dictionary", *pick("dictionary")]
    if pick("conjugations"):
        cmd += ["--conjugations", *pick("conjugations")]
    if pick("qa"):
        cmd += ["--qa", *pick("qa")]
    texts = glob.glob("extracted/*.txt")
    if texts:
        cmd += ["--grammar-text", *texts]
    subprocess.run(cmd, check=True)
    extra = ["--extra-sft", "data/sft_sources.jsonl", "--extra-pretrain", "data/pretrain_sources.jsonl"]
else:
    print("No amharic-sources dataset attached: using public data only.")
extra_args = " ".join(extra)
!python build_corpus.py --out data --max-web-docs {MAX_WEB_DOCS} {extra_args}

## 3. Baseline: the base model before training

In [ ]:
limit = f"--limit {EVAL_LIMIT}" if EVAL_LIMIT else ""
!python eval_amharic.py --model {BASE_MODEL} {limit} --out results/base.json

## 4. Stage A: continued training on Amharic text

In [ ]:
!python train_lora.py --stage pretrain --model {BASE_MODEL} --data data/pretrain.jsonl --out out/stage_a --max-steps {PRETRAIN_STEPS}

## 5. Stage B: instruction tuning on Amharic Q&A

In [ ]:
!python train_lora.py --stage sft --model {BASE_MODEL} --data data/sft.jsonl --eval-data data/sft_eval.jsonl --resume-adapter out/stage_a --out out/stage_b --max-steps {SFT_STEPS}

## 6. Evaluate the trained model and compare

In [ ]:
!python eval_amharic.py --model {BASE_MODEL} --adapter out/stage_b {limit} --out results/tuned.json
base, tuned = json.load(open("results/base.json")), json.load(open("results/tuned.json"))
pd.DataFrame({
    "base": {"Belebele acc": base["belebele"]["accuracy"], "AfriMMLU acc": base["afrimmlu"]["accuracy"]},
    "tuned": {"Belebele acc": tuned["belebele"]["accuracy"], "AfriMMLU acc": tuned["afrimmlu"]["accuracy"]},
})

## 7. Native-speaker review
Read both answers for each prompt and note which is better. Automatic scores do not capture fluency or grammatical correctness; your judgement does.

In [ ]:
for b, t in zip(base["prompts"], tuned["prompts"]):
    print(f"=== {b['id']}. {b['prompt']}")
    if b.get("reference"):
        print(f"Reference: {b['reference']}")
    print(f"--- base:\n{b['answer']}\n--- tuned:\n{t['answer']}\n")

## 8. Save the adapter
The adapter is in `/kaggle/working/tibeb/training/amharic/out/stage_b` (also downloadable from the Output tab). With an `HF_TOKEN` secret it is uploaded as a private model repo.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    try:
        api.create_repo(HUB_ADAPTER_REPO, private=True, exist_ok=True)
        api.upload_folder(folder_path="out/stage_b", repo_id=HUB_ADAPTER_REPO,
                          ignore_patterns=["checkpoint-*"])
        api.upload_folder(folder_path="results", repo_id=HUB_ADAPTER_REPO, path_in_repo="results")
        print("Uploaded to https://huggingface.co/" + HUB_ADAPTER_REPO)
    except Exception as err:
        # e.g. private storage limit reached: free space or set private=False
        print("Upload failed:", err)
else:
    print("No HF_TOKEN secret: adapter kept in /kaggle/working only.")